In [ ]:
"""
================================================================================
TESIS MAGÍSTER EN ECONOMÍA UC
Contagio predictivo sectorial y NBFIs en Chile
================================================================================
Script: grafos_episodios_red.ipynb
Autor:  Carlos González
Fecha:  Abril 2026
Ultima actualizacion: Mayo 2026
"""

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os
import matplotlib.patches as mpatches
from pathlib import Path

# Configuración de tipografía
plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "axes.titlesize":    12,
    "axes.labelsize":    10,
})

# ============================================================
# RUTAS Y CONFIGURACIÓN
# ============================================================
BASE_DIR  = Path("../..") #por favor. Agregar path y luego ejecutar.
INPUT_FILE = BASE_DIR / "1_datos/1_clean_data/formato_excel/panel_red_saldos.xlsx"
OUTPUT_DIR = BASE_DIR / "3_resultados/3_resultados_he_red/grafos/figuras"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

COLORS = {
    "S121":      "#2C3E50",
    "S122":      "#2980B9",
    "S123":      "#27AE60",
    "S124":      "#166757",
    "S125_S126": "#8E44AD",
    "S128":      "#E67E22",
    "S129":      "#AA4B41",
    "S13":       "#7F8C8D",
    "S11":       "#613F06",
    "S14_S15":   "#1ABC9C",
    "S2":        "#74D5E4",
}

LABELS = {
    "S121":      "Banco Central",
    "S122":      "Bancos",
    "S123":      "FMM",
    "S124":      "FNM",
    "S125_S126": "Otros interm.",
    "S128":      "Seguros",
    "S129":      "AFP",
    "S13":       "Gobierno",
    "S11":       "Empresas NF",
    "S14_S15":   "Hogares",
    "S2":        "Resto del mundo",
}

#EVENTS = {
 #   'GCF': {'Antes': '2008T2', 'Durante': '2008T4', 'Post': '2009T3'},
  #  'Estallido': {'Antes': '2019T3', 'Durante': '2019T4', 'Post': '2020T1'},
  # 'Retiros': {'Antes': '2020T2', 'Durante': '2021T2', 'Post': '2022T2'}
#}
EVENTS = {
    'GCF': {'Antes': '2008T2', 'Durante': '2008T4'},
    'Estallido': {'Antes': '2019T3', 'Durante': '2019T4'},
   'Retiros': {'Antes': '2020T2', 'Durante': '2021T2'}
}
# ============================================================
# FUNCIONES PRINCIPALES
# ============================================================

def load_data():
    print("Cargando datos...")
    df = pd.read_excel(INPUT_FILE)
    edges = df.groupby(['periodo', 'sector_activo_codigo', 'sector_pasivo_codigo'])['valor'].sum().reset_index()
    edges = edges[edges['valor'] > 0]
    return edges

def calculate_global_layout(edges):
    print("Calculando posiciones estáticas de los nodos...")
    global_edges = edges.groupby(['sector_activo_codigo', 'sector_pasivo_codigo'])['valor'].mean().reset_index()
    G = nx.DiGraph()
    for _, row in global_edges.iterrows():
        G.add_edge(row['sector_activo_codigo'], row['sector_pasivo_codigo'], weight=row['valor'])
    pos = nx.spring_layout(G, k=3.0, iterations=300, seed=42, weight='weight')
    for k in LABELS.keys():
        if k not in pos: pos[k] = (np.random.rand(), np.random.rand())
    return pos

def draw_network(ax, edges, current_period, base_period, pos, title, global_max_vol, global_max_weight, global_max_delta, is_delta=False):
    df_curr = edges[edges['periodo'] == current_period]
    G = nx.DiGraph()
    for node in LABELS.keys(): G.add_node(node)
        
    if not is_delta:
        # Panel 1: Absoluto
        for _, row in df_curr.iterrows():
            u, v, w = row['sector_activo_codigo'], row['sector_pasivo_codigo'], row['valor']
            if u in LABELS and v in LABELS:
                G.add_edge(u, v, weight=w, color=COLORS.get(u, '#7f8c8d'))
    else:
        # Paneles 2 y 3: Deltas (Variación vs Antes)
        df_base = edges[edges['periodo'] == base_period]
        df_merge = pd.merge(df_curr, df_base, on=['sector_activo_codigo', 'sector_pasivo_codigo'], how='outer', suffixes=('_curr', '_base')).fillna(0)
        df_merge['delta'] = df_merge['valor_curr'] - df_merge['valor_base']
        
        # Filtramos variaciones menores al 2% del flujo maximo del evento para quitar ruido
        threshold = global_max_weight * 0.02 
        
        for _, row in df_merge.iterrows():
            u, v, d = row['sector_activo_codigo'], row['sector_pasivo_codigo'], row['delta']
            if abs(d) > threshold and u in LABELS and v in LABELS:
                color = COLORS.get(u, '#7f8c8d') # Mantener color del sector origen
                style = 'solid' if d > 0 else 'dashed' # Linea continua = Aumento, Punteada = Caída
                G.add_edge(u, v, weight=abs(d), color=color, style=style)

    # Tamaños de nodos proporcionales al volumen absoluto en ESE momento (para dar contexto del tamaño del sector)
    node_vols = {}
    for node in G.nodes():
        v_in = df_curr[df_curr['sector_pasivo_codigo'] == node]['valor'].sum() if not df_curr.empty else 0
        v_out = df_curr[df_curr['sector_activo_codigo'] == node]['valor'].sum() if not df_curr.empty else 0
        node_vols[node] = v_in + v_out
        
    base_size = 250
    node_sizes = [base_size + (node_vols[node] / global_max_vol) * 1500 if global_max_vol > 0 else base_size for node in G.nodes()]
    node_colors = [COLORS.get(node, '#CCCCCC') for node in G.nodes()]
    labels = {node: LABELS.get(node, node) for node in G.nodes()}
    
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes, node_color=node_colors, edgecolors='white', linewidths=1.5, alpha=0.9)
    
    edge_weights = nx.get_edge_attributes(G, 'weight')
    edge_colors_list = nx.get_edge_attributes(G, 'color')
    
    if edge_weights and global_max_weight > 0:
        if not is_delta:
            widths = [(w / global_max_weight) * 5.0 + 0.5 for w in edge_weights.values()]
            colors = list(edge_colors_list.values())
            nx.draw_networkx_edges(G, pos, ax=ax, width=widths, edge_color=colors, alpha=0.8, arrowsize=20, arrowstyle='-|>', connectionstyle='arc3, rad=0.15', style='solid')
        else:
            # Para deltas, separamos las aristas según el estilo (aumento vs caída) para dibujarlas correctamente
            edges_solid = [(u, v) for u, v, d in G.edges(data=True) if d['style'] == 'solid']
            edges_dashed = [(u, v) for u, v, d in G.edges(data=True) if d['style'] == 'dashed']
            
            if edges_solid:
                widths_s = [(G[u][v]['weight'] / global_max_delta) * 5.0 + 0.5 for u, v in edges_solid]
                colors_s = [G[u][v]['color'] for u, v in edges_solid]
                nx.draw_networkx_edges(G, pos, edgelist=edges_solid, ax=ax, width=widths_s, edge_color=colors_s, alpha=0.8, arrowsize=20, arrowstyle='-|>', connectionstyle='arc3, rad=0.15', style='solid')
                
            if edges_dashed:
                widths_d = [(G[u][v]['weight'] / global_max_delta) * 5.0 + 0.5 for u, v in edges_dashed]
                colors_d = [G[u][v]['color'] for u, v in edges_dashed]
                nx.draw_networkx_edges(G, pos, edgelist=edges_dashed, ax=ax, width=widths_d, edge_color=colors_d, alpha=0.8, arrowsize=20, arrowstyle='-|>', connectionstyle='arc3, rad=0.15', style='dashed')
        
    pos_labels = {k: (v[0], v[1]-0.12) for k, v in pos.items()}
    nx.draw_networkx_labels(G, pos_labels, labels=labels, ax=ax, font_size=8, font_family='serif', font_weight='bold')
    
    ax.set_title(title, pad=10)
    ax.axis('off')
    
    if is_delta:
        from matplotlib.lines import Line2D
        solid_line = Line2D([0], [0], color='gray', linewidth=2, linestyle='solid', label='Aumento Flujo')
        dashed_line = Line2D([0], [0], color='gray', linewidth=2, linestyle='dashed', label='Caída Flujo')
        ax.legend(handles=[solid_line, dashed_line], loc='lower right', fontsize=9)

def create_event_figure(event_name, moments, edges, pos):
    print(f"Generando figura para: {event_name}")
    
    # Calcular volumen maximo a nivel de evento para normalizar
    node_vols_event = []
    for period in moments.values():
        df_p = edges[edges['periodo'] == period]
        for node in LABELS.keys():
            v_in = df_p[df_p['sector_pasivo_codigo'] == node]['valor'].sum()
            v_out = df_p[df_p['sector_activo_codigo'] == node]['valor'].sum()
            node_vols_event.append(v_in + v_out)
    global_max_vol = max(node_vols_event) if node_vols_event else 1
    
    periods_list = list(moments.values())
    global_max_weight = edges[edges['periodo'].isin(periods_list)]['valor'].max()
    
    # Calcular max delta para escalar los grosores de variacion
    base_period = moments['Antes']
    df_base = edges[edges['periodo'] == base_period]
    global_max_delta = 1
    for p in list(moments.values())[1:]:
        df_curr = edges[edges['periodo'] == p]
        df_merge = pd.merge(df_curr, df_base, on=['sector_activo_codigo', 'sector_pasivo_codigo'], how='outer', suffixes=('_curr', '_base')).fillna(0)
        df_merge['delta'] = df_merge['valor_curr'] - df_merge['valor_base']
        current_max_delta = df_merge['delta'].abs().max()
        if current_max_delta > global_max_delta:
            global_max_delta = current_max_delta
    
    #fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f"Red de Interconexiones - {event_name}", fontsize=15, y=1.05)
    #fig.suptitle(f"Red de Interconexiones - {event_name}\n(Paneles 2 y 3 muestran la variación de flujos respecto a 'Antes')", fontsize=15, y=1.05)
    
    base_period = moments['Antes']
    
    for i, (moment_name, period) in enumerate(moments.items()):
        ax = axes[i]
        is_delta = (i > 0)
        title = f"{moment_name} ({period})" if not is_delta else f"Variación: {moment_name} ({period}) vs Antes"
        draw_network(ax, edges, period, base_period, pos, title, global_max_vol, global_max_weight, global_max_delta, is_delta)
        
    plt.tight_layout()
    output_path = os.path.join(OUTPUT_DIR, f"fig_redes_{event_name}_Deltas.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return output_path

if __name__ == "__main__":
    edges_df = load_data()
    pos = calculate_global_layout(edges_df)
    
    for event_name, moments in EVENTS.items():
        output_path = create_event_figure(event_name, moments, edges_df, pos)
        print(f"✅ Figura generada en: {output_path}")

Cargando datos...
Calculando posiciones estáticas de los nodos...
Generando figura para: GCF
✅ Figura generada en: ../../3_resultados/3_resultados_he_red/grafos/figuras/fig_redes_GCF_Deltas.png
Generando figura para: Estallido
✅ Figura generada en: ../../3_resultados/3_resultados_he_red/grafos/figuras/fig_redes_Estallido_Deltas.png
Generando figura para: Retiros
✅ Figura generada en: ../../3_resultados/3_resultados_he_red/grafos/figuras/fig_redes_Retiros_Deltas.png
